In [ ]:
import os
import netCDF4 as nc
import pandas as pd
import numpy as np
import numpy.ma as ma
import xarray as xr
import netCDF4 as nc
from netCDF4 import Dataset
import shapefile
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import time
import sys
import pickle
from datetime import timedelta
import catboost as cb
# 设置参数
OUTPUT_FOLDER = "default"
CAL_IMP_METHOD = "shap" # 计算特征重要性的方法，包括：perm、MDI、shap
feature_selection_strategy = "RFE"
SCALE_FACTOR = 1
NCPU = 30
SCORE="KGE"
BASIN_NAME = "CN"
SOIL_DEPTH = "10cm"
TARGET = f'OBS_ST_{SOIL_DEPTH}'
if TARGET == "OBS_SKT":
    # ============================== SKT =================================
    SOIL_DEPTH = "0cm"
    START_TEST_DATE = "2012-01-01"
    TRAIN_DATA_FILENAME = f'SKTmerge_daily_10cm_1km_train_by_Date_{START_TEST_DATE}.parquet'
    VALIDATION_DATA_FILENAME = f'SKTmerge_daily_10cm_1km_valid_by_Date_{START_TEST_DATE}.parquet'
    TARGET = f'OBS_SKT'
    # =============================================================================
elif TARGET == "OBS_ST_10cm":
    # ============================== ST =================================
    SOIL_DEPTH = "10cm"
    TRAIN_DATA_FILENAME = f"STmerge_daily_OBS_ST_10cm_1km_train_by_Date_2013-01-01.parquet"
    VALIDATION_DATA_FILENAME = f"STmerge_daily_OBS_ST_10cm_1km_valid_by_Date_2013-01-01.parquet"
    TARGET = f'OBS_ST_{SOIL_DEPTH}'
# 设置路径
BASE_PATH = "/home/yfdong/data/work/STmerge/v1.0/"
DATA_PATH = "/raid61/yfdong/data/work/STmerge/v1.0"
DB_PATH = os.path.join(DATA_PATH , "dataframe/database")
SAVE_PATH = os.path.join(DATA_PATH , "dataframe/train_output", SOIL_DEPTH, OUTPUT_FOLDER)
FEATURE_PATH = os.path.join(SAVE_PATH, 'FeatureSelection')
OPTUNA_PATH = os.path.join(SAVE_PATH, "Optuna")

In [ ]:
# 输出路径
print(f"Save path: {SAVE_PATH}")
print(f"Feature path: {FEATURE_PATH}")
print(f"Optuna path: {OPTUNA_PATH}")

In [ ]:
# sys.path.append(f"{BASE_PATH}/code/Library")
sys.path.append(f"{BASE_PATH}/code/Library")
from MergeST import save_files, save_model_scaler, calculate_metrics, get_DEFmodel, get_OPTUNAmodel, preprocess_features, train_model, evaluate_model_byCV, load_data, load_data_ByStratifiedSampling
train_data = load_data(TRAIN_DATA_FILENAME, DB_PATH)
valid_data = load_data(VALIDATION_DATA_FILENAME, DB_PATH)
total_data = pd.concat([train_data, valid_data], axis=0, ignore_index=True)

In [ ]:
print(f"Train data shape: {train_data.shape}")
print(f"Validation data shape: {valid_data.shape}")
print(f"Total data shape: {total_data.shape}")

In [ ]:
TotalFeatures = pd.read_csv(os.path.join(DB_PATH, 'TotalFeatureColumns.csv'))["Feature"].tolist()
SubsetFeatures = pd.read_csv(os.path.join(FEATURE_PATH, f'RFE_shap_CB_CN_{TARGET}_subset_feature.csv'))["Feature"].tolist()
print("TotalFeature nums:", len(TotalFeatures), "TotalFeatures:", TotalFeatures)

In [ ]:
def train_model_byPool(train_pool, model):
    start_train = time.time()
    model.fit(train_pool)
    end_train = time.time()
    training_time_formatted= str(timedelta(seconds=end_train - start_train))
    print(f"Training completed in {training_time_formatted}")
    return model

In [ ]:
# ================================================== 利用默认的参数训练ML模型 ==================================================
MODELS = ["CB"]

# Preprocess data
X_train_scaled, y_train, scaler = preprocess_features(train_data, TotalFeatures, TARGET, SCALE_FACTOR)
train_pool = cb.Pool(data=X_train_scaled, label=y_train, feature_names=TotalFeatures)

start = time.time()
print(f"#----------------{SOIL_DEPTH}---------------#")
print(f"*******************{BASIN_NAME}*******************")
for MODEL_NAME in MODELS:
    print(f"#----------------{MODEL_NAME}---------------#")
    # =============================================================================
    # TEST_COMPARISON_FILENAME = f"SM_comparison_valid.csv"
    print("SAVE_PATH", SAVE_PATH)
    DEFmodel = get_DEFmodel(MODEL_NAME, n_cpu=NCPU,SimpleModel=False)
    
    # Train and cross-validation model
    print(f"start train ML {MODEL_NAME} model based on default hyperparameter combination ")
    StartTime = time.time()
    DEFmodel = train_model_byPool(train_pool, DEFmodel)
    EndTime = time.time()
    
    # Save model and results
    save_model_scaler(os.path.join(SAVE_PATH, "model"), f"{MODEL_NAME}_{TARGET}_DEF_allFeature", DEFmodel, scaler)
    print(f"Save the ML model {MODEL_NAME} based on default hyperparameter combination to {SAVE_PATH}")
    
    # Predict and evaluate on train set
    print(f"start predicte ML model {MODEL_NAME} based on default hyperparameter combination ......")
    StartTime = time.time()
    X_valid = valid_data[TotalFeatures]
    y_valid = valid_data[TARGET] * SCALE_FACTOR
    X_valid_scaled = scaler.transform(X_valid)
    DEFpredicted_valid = DEFmodel.predict(X_valid_scaled)
    EndTime = time.time()
    TimeFormatted= str(timedelta(seconds=EndTime - StartTime))
    print(f"Predicte the ML model {MODEL_NAME} took {TimeFormatted}. Testing Set Size: {len(X_valid_scaled)} .")
    
    # Calculate metrics
    print(f"*******************DEF************************")
    calculate_metrics(DEFpredicted_valid, y_valid)

end = time.time()
print(f"Elapsed Time: {end - start} seconds")

